In [106]:
import sys
import yaml
import pandas as pd

sys.path.insert(0, '/home/markakis/chunkbench/src')

from autoslo.filesystem.structured_log import StructuredLog
from autoslo.slo.slo_resolver import SloResolver
from autoslo.config.component_configs import SloResolverConfig

path = './structured_log.parquet'

exec_cfg_path = '/home/markakis/chunkbench/data/runs/1783260865180/execution_config.yml'
with open(exec_cfg_path) as f:
    exec_cfg = yaml.safe_load(f)

slo_resolver_config = SloResolverConfig(**exec_cfg['slo_resolver_config'])

resolver = SloResolver(slo_resolver_config)
slog = StructuredLog.load(path)
df = slog.logos_df(slo_resolver=resolver)


In [93]:
df

,wall_clock_s,rel_time_s,source,event_type,query_id,query_text_id,cluster_name,reason,workload_name,num_queries,...,used,reserved,available,final_latency_s,slo_s,slo_violated,slo_overshoot_s,relative_violation,actual_execution_latency_s,predicted_latency_s
0,1.783261e+09,0.000000,ManagedClusterPool,spin_up_requested,None,None,,initial,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.783261e+09,0.000000,RedshiftServerlessProvisioner,spin_up_started,None,None,autoslo-32-1783260865180-0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.783261e+09,0.000000,ManagedClusterPool,cluster_ready,None,None,autoslo-32-1783260865180-0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.783261e+09,-29.999977,WorkloadRunner,run_start,None,None,,NaN,redbench_provisioned_157_0,1392.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.783261e+09,0.030367,WorkloadRunner,arrival,204510,ext_tpcds1000#009#003,,NaN,NaN,NaN,...,NaN,NaN,NaN,19.501,45.9395,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21375,1.783276e+09,14045.799569,ManagedClusterPool,cluster_removed,None,None,autoslo-16-1783260865180-6,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21376,1.783276e+09,14045.800094,ManagedClusterPool,cluster_removed,None,None,autoslo-16-1783260865180-8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21377,1.783276e+09,14045.800658,ManagedClusterPool,cluster_removed,None,None,autoslo-32-1783260865180-0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21378,1.783276e+09,14045.802009,ManagedClusterPool,cluster_removed,None,None,autoslo-16-1783260865180-7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
from logos import Logos

In [108]:
per_unit = [
    "slo_violated", "slo_s", "slo_overshoot_s", "relative_violation",
    "final_latency_s", "actual_execution_latency_s",
    "selected_cluster_name", "selected_rpu", "prediction_error",
]

lg = Logos.from_parsed_table(
    data=df,
    workdir=".",
    source_id="structured_log_p2",
    template_col="event_type",
    passthrough_cols=["query_id", "query_text_id"],
    per_unit_cols=per_unit,
)


In [96]:
lg.parsed_templates

,TemplateId,TemplateText,Occurrences,VariableIndices,RegexIndices


In [97]:
lg.parsed_variables

,Name,Tag,TagOrigin,Type,IsUninteresting,Occurrences,Preceding 3 tokens,Examples,From regex
0,2f4a6473_0,wall_clock_s,4,num,False,21380,[],"[1783260865.685575, 1783260865.686137, 1783261...",True
1,9bed2068_0,rel_time_s,4,num,False,21380,[],"[0.0, -29.999977111816406, 0.03036689758300781...",True
2,36cd38f4_0,source,4,str,False,21380,[],"[ManagedClusterPool, RedshiftServerlessProvisi...",True
3,1cd03614_0,event_type,4,str,False,21380,[],"[spin_up_requested, spin_up_started, cluster_r...",True
4,0bbeda9c_0,query_id,4,str,False,21236,[],"[204510, 217123, 193438, 215584, 209512]",True
5,28f4a0f8_0,query_text_id,4,str,False,21236,[],"[ext_tpcds1000#009#003, ext_tpcds1000#009#001,...",True
6,5e2a4388_0,cluster_name,4,str,False,21380,[],"[, autoslo-32-1783260865180-0, no-spinup-basel...",True
7,40bea8d6_0,reason,4,str,False,41,[],"[initial, observation_window_start_s=3108.0634...",True
8,db137bf8_0,workload_name,4,str,False,2,[],[redbench_provisioned_157_0],True
9,7c908f5b_0,num_queries,4,num,False,1,[],[1392.0],True


In [109]:
lg.set_causal_unit("query_id")
custom_imp = {tag: "zero_imp" for tag in lg.parsed_variables["Tag"]}
lg.prepare(force=True, custom_imp=custom_imp)


Imputing missing values...:   0%|          | 0/180 [00:00<?, ?it/s]

One-hot encoding categorical variables...:   0%|          | 0/21 [00:00<?, ?it/s]

In [99]:
lg.prepared_log

,2f4a6473_0+mean,9bed2068_0+mean,81046e28_0+mean,d6dd829e_0+mean,4e1566f0_0+mean,2abc53a8_0+mean,2b91dc2f_0+mean,260ca9dd_0+mean,ff954a61_0+mean,86a6fe79_0+mean,...,5e2a4388_0+last=autoslo-16-1783260865180-5,5e2a4388_0+last=autoslo-16-1783260865180-6,5e2a4388_0+last=autoslo-16-1783260865180-7,5e2a4388_0+last=autoslo-16-1783260865180-8,5e2a4388_0+last=autoslo-16-1783260865180-9,5e2a4388_0+last=autoslo-32-1783260865180-0,5e2a4388_0+last=autoslo-32-1783260865180-1,5e2a4388_0+last=autoslo-32-1783260865180-2,7ee7184c_0+last=0,7ee7184c_0+last=pre_spinup
0bbeda9c_0+last,,,,,,,,,,,,,,,,,,,,,
173857,1.783267e+09,6029.974639,13.662600,0.000000,7.217000,0.0,22.254421,1.0,0.0,17.804,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
173858,1.783272e+09,10717.262371,0.642222,0.044444,30.563000,0.0,0.666484,1.0,0.0,0.730,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
173944,1.783271e+09,10159.216320,1.449333,0.175889,26.323444,0.0,0.000000,1.0,0.0,4.005,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
173945,1.783271e+09,9485.830531,0.859889,0.018556,19.840333,0.0,0.000000,1.0,0.0,0.863,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
173946,1.783266e+09,5006.336688,7.935000,0.000000,4.647000,0.0,0.000000,1.0,0.0,5.419,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221123,1.783268e+09,7165.383614,27.465625,0.017875,11.709625,0.0,37.612596,1.0,0.0,62.342,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
221158,1.783272e+09,11184.885021,2.926778,0.000000,32.200000,0.0,0.000000,1.0,0.0,15.078,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
221166,1.783271e+09,9580.697913,4.662667,0.268222,20.978778,0.0,12.000365,1.0,0.0,10.187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [100]:
lg.prepared_variables

,Name,Base,Pre-agg Value,Agg,Post-agg Value,Tag,Base Variable Occurences,Type,Examples,From regex,TemplateText
0,2f4a6473_0+mean,2f4a6473_0,,mean,,wall_clock_s mean,21380,num,"[1783260865.685575, 1783260865.686137, 1783261...",True,
1,9bed2068_0+mean,9bed2068_0,,mean,,rel_time_s mean,21380,num,"[0.0, -29.999977111816406, 0.03036689758300781...",True,
2,81046e28_0+mean,81046e28_0,,mean,,latency_s_for_routing mean,11536,num,"[15.494, 17.444, 17.443, 23.051, 26.054]",True,
3,d6dd829e_0+mean,d6dd829e_0,,mean,,slo_violation mean,11576,num,"[0.0, 0.5, 0.167, 0.143, 0.25]",True,
4,4e1566f0_0+mean,4e1566f0_0,,mean,,cost mean,11576,num,"[0.0, 0.2, 0.4, 0.6, 0.8]",True,
...,...,...,...,...,...,...,...,...,...,...,...
99,5e2a4388_0+last=autoslo-32-1783260865180-0,5e2a4388_0,,last,autoslo-32-1783260865180-0,cluster_name last autoslo-32-1783260865180-0,21380,str,"[, autoslo-32-1783260865180-0, no-spinup-basel...",True,
100,5e2a4388_0+last=autoslo-32-1783260865180-1,5e2a4388_0,,last,autoslo-32-1783260865180-1,cluster_name last autoslo-32-1783260865180-1,21380,str,"[, autoslo-32-1783260865180-0, no-spinup-basel...",True,
101,5e2a4388_0+last=autoslo-32-1783260865180-2,5e2a4388_0,,last,autoslo-32-1783260865180-2,cluster_name last autoslo-32-1783260865180-2,21380,str,"[, autoslo-32-1783260865180-0, no-spinup-basel...",True,
102,7ee7184c_0+last=0,7ee7184c_0,,last,0,phase last 0,72,str,[pre_spinup],True,


In [110]:
lg.rank_candidate_causes('slo_violated mean', prune_candidates=False)


,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,99854374_0+mean,slo_s mean,slo_violated mean,-2.267603e-02,4.001640e-09,Undecided,Undecided
1,17060c6a_0+mean,prediction_error mean,slo_violated mean,7.938211e-03,1.902651e-06,Undecided,Undecided
2,c1357543_0+mean,slo_overshoot_s mean,slo_violated mean,8.954239e-03,3.087967e-06,Undecided,Undecided
3,5bfc897d_0+mean,wall_clock_s mean,slo_violated mean,1.380353e-10,2.271550e-05,Undecided,Undecided
4,8bbdd445_0+mean,wall_clock_s mean,slo_violated mean,1.380353e-10,2.271550e-05,Undecided,Undecided
...,...,...,...,...,...,...,...
122,3609b411_2+last=WorkloadRunner,source last WorkloadRunner,slo_violated mean,NaN,NaN,Undecided,Undecided
123,8bbdd445_2+last=WorkloadRunner,source last WorkloadRunner,slo_violated mean,NaN,NaN,Undecided,Undecided
124,5bfc897d_2+last=WorkloadRunner,source last WorkloadRunner,slo_violated mean,NaN,NaN,Undecided,Undecided
125,723a9a91_2+last=Autoscaler,source last Autoscaler,slo_violated mean,NaN,NaN,Undecided,Undecided


In [111]:
lg.rank_candidate_causes('prediction_error mean', prune_candidates=False)


,Candidate,Candidate Tag,Target Tag,Slope,P-value,Candidate->Target Edge Status,Target->Candidate Edge Status
0,86a6fe79_0+mean,final_latency_s mean,prediction_error mean,0.946968,2.147100e-35,Undecided,Undecided
1,0b37e060_0+mean,actual_execution_latency_s mean,prediction_error mean,0.946957,2.164646e-35,Undecided,Undecided
2,c1357543_0+mean,slo_overshoot_s mean,prediction_error mean,1.080892,1.668917e-31,Undecided,Undecided
3,ef3b209b_3+last=autoslo-16-1783260865180-6,cluster_name last autoslo-16-1783260865180-6,prediction_error mean,105.197590,9.838952e-19,Undecided,Undecided
4,133bdb9d_3+last=autoslo-16-1783260865180-6,cluster_name last autoslo-16-1783260865180-6,prediction_error mean,105.197590,9.838952e-19,Undecided,Undecided
...,...,...,...,...,...,...,...
122,3609b411_2+last=WorkloadRunner,source last WorkloadRunner,prediction_error mean,NaN,NaN,Undecided,Undecided
123,8bbdd445_2+last=WorkloadRunner,source last WorkloadRunner,prediction_error mean,NaN,NaN,Undecided,Undecided
124,5bfc897d_2+last=WorkloadRunner,source last WorkloadRunner,prediction_error mean,NaN,NaN,Undecided,Undecided
125,723a9a91_2+last=Autoscaler,source last Autoscaler,prediction_error mean,NaN,NaN,Undecided,Undecided
